# Project 4: **Build a Deep Research System**
Welcome to project 4! For this project, we shift our focus from tool use and agents to *reasoning* models. You will practice state‑of‑the‑art inference‑time scaling methods such as *Chain‑of‑Thought* prompting and *Tree‑of‑Thoughts*, and briefly explore high-levels of training reasoning models using techniques like **STaR**.


Finally, you will put everything together to build a *deep research agent* that can browse the web, reason over what it finds, and give structured answers.

## Learning Objectives  
* Apply common inference‑time scaling methods: **zero‑shot / few‑shot CoT, self‑consistency, sequential decoding, tree‑of‑thoughts**  
* Gain intuition for **training** reasoning‑capable models following **STaR** approach 
* Build a minimal **deep‑research agent** that combines step‑by‑step reasoning with live web search   
* Practice extending deep-search to a multi-agent system 

## Roadmap  
1. Environment setup  
2. Inference‑time scaling  
   2.1 Few‑shot & zero‑shot CoT  
   2.2 Self‑consistency
   2.3 Sequential revisions  
   2.4 Tree‑of‑Thought
3. STaR for training models for reasoning  
4. Deep-research agent  
5. (Optional) Multi-agent deep-research

# 1‑ Environment setup

## 1.1- Conda environment

Before we start coding, you need a reproducible setup. Open a terminal in the same directory as this notebook and run:

```bash
# Create and activate the conda environment
conda env create -f environment.yaml && conda activate deep_research

# Register this environment as a Jupyter kernel
python -m ipykernel install --user --name=deep_research --display-name "deep_research"
```
Once this is done, you can select "deep_research" from the Kernel → Change Kernel menu in Jupyter or VS Code.

## 1.2 Ollama setup

In this project we use the `llama3.2:3b` and `deepseek-r1:8b` models. You can try other smaller or larger reasoning LLMs such as `qwen2.5:3b-instruct` or `phi4-mini` to compare performance. Explore available models here: https://ollama.com/library.

```bash
ollama pull llama3.2:3b
ollama pull deepseek-r1:8b
# Additional small reasoning models to compare
# ollama pull qwen2.5:3b-instruct
# ollama pull phi4-mini

```

`ollama pull` downloads the model so you can run it locally without API calls.

---  
# 2‑ Inference‑time scaling

Inference-time scaling refers to techniques that make an existing model reason better without retraining it. Instead of changing the model’s weights, we achieve reasoning capability by adjusting how we prompt, sample, or aggregate LLM's outputs.

In this section, we’ll explore several inference-time strategies that improve reasoning quality using a non-reasoning base model. You will experiment with and compare methods such as:

- Few-shot Chain-of-Thought (CoT)
- Zero-shot CoT
- Self-consistency
- Sequential revision
- Tree-of-Thoughts (ToT)

### 2.1: Few‑Shot CoT
Few-shot prompting helps a model reason by showing one or multiple examples before asking a new question. By observing the pattern of reasoning and final answers, the model learns how to structure its own reasoning process on the new input.

In this exercise, you will create a prompt that includes a few example Q&A pairs demonstrating step-by-step reasoning. Then, you will feed a new question and see the model’s output.

In [1]:
# Step 1: Write a few examples showing reasoning steps
# Step 2: Write your new question
# Step 3: Concatenate examples + new question into a single prompt
# Step 4: Call your Ollama or OpenAI client to get a response from llama3.2:3b # e.g., client.chat.completions.create(...)
# Step 5: Print the final answer

from openai import OpenAI

"""
YOUR CODE HERE (~10 lines of code)
"""

# Connect to Ollama
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

# Step 1: Few-shot reasoning examples
examples = """
Q: Tom has 3 apples and buys 2 more. How many apples does he have?
A: Tom starts with 3 apples. He buys 2 more. Total = 3 + 2 = 5.

Q: A train travels 60 km in 1 hour. How far will it travel in 4 hours?
A: Distance = Speed × Time = 60 × 4 = 240 km.
"""

# Step 2: New question
question = "Q: Sarah has 12 candies. She gives 5 to her friend and then buys 8 more. How many candies does she have now?\nA:"

# Step 3: Build the prompt
prompt = examples + "\n" + question

# Step 4: Generate a response
response = client.chat.completions.create(
    model="gemma4:e2b",
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0
)

# Step 5: Print the answer
print(response.choices[0].message.content)

Sarah starts with 12 candies.
She gives 5 to her friend: $12 - 5 = 7$
She then buys 8 more: $7 + 8 = 15$

**Answer:** Sarah has 15 candies now.


### (Optional) Few-shot CoT on GPT2
GPT-2 is a pre-trained language model without instruction tuning. It continues text rather than answering questions. In this section, you'll try the exact same CoT pattern on GPT-2 and observe what happens. The goal is to test whether few-shot CoT alone can elicit structured reasoning from a non-chat LLM.

In [2]:
import os
import torch
from transformers import pipeline

# Step 1: Load GPT-2 text-generation from huggingface (https://huggingface.co/docs/transformers/en/model_doc/gpt2)
# Step 2: Write 1–2 few-shot reasoning examples (short, explicit steps + final answer in your own unique format)
# Step 3: Append a new test question after the examples to form one prompt string
# Step 4: Generate 1–3 completions with different decoding settings (e.g., greedy vs. top-k)
# Step 5: Print raw outputs; check if steps are followed and if the final answer is correct

# Step 1: Load GPT-2 text generation pipeline
generator = pipeline(
    "text-generation",
    model="gpt2",
    device=0 if torch.cuda.is_available() else -1
)

# Step 2 & 3: Few-shot examples + new question
prompt = """
Example 1
Question: John has 5 pens and buys 3 more.
Reasoning: Start with 5. Add 3. Total = 8.
Answer: 8

Example 2
Question: A box has 10 oranges. 4 are removed.
Reasoning: Start with 10. Subtract 4. Total = 6.
Answer: 6

Example 3
Question: Sarah has 12 candies. She gives away 5 and buys 8 more.
Reasoning:
"""

# Step 4: Generate with different decoding settings
greedy = generator(
    prompt,
    max_new_tokens=50,
    do_sample=False
)

topk = generator(
    prompt,
    max_new_tokens=50,
    do_sample=True,
    top_k=50,
    temperature=0.8
)

# Step 5: Print outputs
print("=== Greedy Output ===")
print(greedy[0]["generated_text"])

print("\n=== Top-k Sampling Output ===")
print(topk[0]["generated_text"])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'top_k', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both

=== Greedy Output ===

Example 1
Question: John has 5 pens and buys 3 more.
Reasoning: Start with 5. Add 3. Total = 8.
Answer: 8

Example 2
Question: A box has 10 oranges. 4 are removed.
Reasoning: Start with 10. Subtract 4. Total = 6.
Answer: 6

Example 3
Question: Sarah has 12 candies. She gives away 5 and buys 8 more.
Reasoning:

Start with 12. Subtract 4. Total = 8.

Answer: 8

Example 4

Question: A box has 10 oranges. 4 are removed.

Reasoning:

Start with 10. Subt

=== Top-k Sampling Output ===

Example 1
Question: John has 5 pens and buys 3 more.
Reasoning: Start with 5. Add 3. Total = 8.
Answer: 8

Example 2
Question: A box has 10 oranges. 4 are removed.
Reasoning: Start with 10. Subtract 4. Total = 6.
Answer: 6

Example 3
Question: Sarah has 12 candies. She gives away 5 and buys 8 more.
Reasoning:

Starting 10 is the most efficient way to start your list, so your list will continue to grow.

Example 4

Question: A box has 20 oranges. 11 are removed.

Reasoning: Start with 10.

### 2.2: Zero‑Shot Chain‑of‑Thought
Zero-shot CoT encourages the model to reason without examples by adding a short cue such as “Let’s think step by step.” This simple phrase often activates the model’s latent reasoning ability even when no demonstrations are provided. It serves as a baseline to compare with few-shot and other inference-time scaling methods.

In [3]:
from openai import OpenAI

# Step 1: Write the question and a zero-shot CoT cue (e.g., "Let's think step by step.")
# Step 2: Build a single prompt string that includes brief role guidance plus the question
# Step 3: Call your Ollama or OpenAI client to get a response from llama3.2:3b  # e.g., client.chat.completions.create(...)
# Step 4: Print the chain and the final answer

# Connect to Ollama
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

# Step 1 & 2: Question with zero-shot CoT cue
prompt = (
    "You are a helpful reasoning assistant.\n\n"
    "Question: A store has 25 books. It sells 9 books and receives 14 new books. "
    "How many books are in the store now?\n"
    "Let's think step by step."
)

# Step 3: Generate response
response = client.chat.completions.create(
    model="llama3.2:latest",
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0
)

# Step 4: Print reasoning and final answer
print(response.choices[0].message.content)

To find out how many books are in the store now, we need to follow these steps:

1. Start with the initial number of books: The store has 25 books.
2. Subtract the number of books sold: The store sells 9 books, so we subtract 9 from 25:
   25 - 9 = 16
3. Add the new books received: The store receives 14 new books, so we add 14 to 16:
   16 + 14 = 30

Therefore, there are now 30 books in the store.


### 2.3 Self‑Consistency
Self-consistency enhances reasoning accuracy by sampling multiple independent reasoning paths for the same question instead of relying on a single deterministic answer. Each run may follow a slightly different logical chain, and the diversity helps correct individual mistakes. After generating several reasoning traces, you then aggregate the final answers using majority voting.

This approach is especially useful when tasks involve multi-step reasoning or arithmetic, where single-path outputs may be incorrect.

In [4]:
from openai import OpenAI
import re, collections

client = OpenAI(api_key = "ollama", base_url = "http://localhost:11434/v1")
MODEL = "llama3.2:latest"

def cot_answer(question, temperature=1.0):
    # Generate a step-by-step reasoning chain for the given question and extract the final answer.
    """
    YOUR CODE HERE (~10 lines of code)
    """
    # Generate a step-by-step reasoning chain
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": f"{question}\n\nLet's think step by step and end with 'Final Answer: <answer>'."
            }
        ],
        temperature=temperature,
    )

    text = response.choices[0].message.content

    # Extract final answer
    match = re.search(r"Final Answer:\s*(.+)", text, re.IGNORECASE)
    if match:
        answer = match.group(1).strip()
    else:
        # Fallback: use the last number found
        nums = re.findall(r"-?\d+\.?\d*", text)
        answer = nums[-1] if nums else text.strip()

    return text, answer

def self_consistent(question, n=10):
    # Run multiple reasoning chains and select the most frequent final answer by majority voting.
    """
    YOUR CODE HERE (~12 lines of code)
    """
    # Generate multiple reasoning paths
    answers = []

    for _ in range(n):
        _, ans = cot_answer(question, temperature=1.0)
        answers.append(ans)

    # Majority voting
    counter = collections.Counter(answers)
    winner = counter.most_common(1)[0][0]

    return winner, counter


question = "What is the square root of 144?"
winner, counter = self_consistent(question)
print("Votes:", counter)
print("Chosen answer:", winner)

Votes: Counter({'12': 6, '$\\boxed{12}$': 1, '2': 1, '√144= 12': 1, '$\\\\boxed{12}$': 1})
Chosen answer: 12


### 2.4: Sequential Revision

Sequential revision iteratively improves an answer by generating a first draft, critiquing it, and producing revised drafts that condition on prior answers. Each round should be short and focused, so improvements accumulate without drifting from the question.

In [5]:
from openai import OpenAI

client = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)

MODEL = "llama3.2:latest"


def sequential_revision(question: str, max_steps: int = 3) -> str:
    # Step 1: Generate the initial draft
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": f"Answer the following question:\n\n{question}"}
        ],
        temperature=0.7,
    )

    draft = response.choices[0].message.content
    print(f"\n--- Draft 1 ---\n{draft}")

    # Step 2: Iteratively revise the answer
    for step in range(2, max_steps + 1):
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {
                    "role": "user",
                    "content": (
                        f"Question:\n{question}\n\n"
                        f"Current Answer:\n{draft}\n\n"
                        "Revise this answer to make it more accurate, "
                        "clear, and concise while preserving correct information."
                    ),
                }
            ],
            temperature=0.7,
        )

        draft = response.choices[0].message.content
        print(f"\n--- Draft {step} ---\n{draft}")

    # Step 4: Return the final draft
    return draft


# Step 1: Define a reasoning question
question = (
    "A farmer has 17 sheep. All but 9 die. "
    "How many sheep are left? Explain your reasoning."
)

# Step 2 & 3: Run sequential revision and print final output
final_answer = sequential_revision(question, max_steps=3)

print("\n=== Final Improved Answer ===")
print(final_answer)


--- Draft 1 ---
The statement "all but 9" means that 9 sheep did not die. Since all the other sheep died, the only ones left are the 9 that "all but 9" refers to.

So, there are 9 sheep left.

--- Draft 2 ---
Here's a revised explanation:

The phrase "all but 9" means that all of the sheep except for 9 died. This is not an indication that the remaining 9 sheep did not die; rather, it signifies that there were initially 9 sheep that survived.

Since 17 sheep were lost and only 9 remained alive at some point, the correct interpretation is to subtract the number of sheep that died from the total initial number of sheep. 

The calculation would be: 
17 (initial sheep) - x (number of sheep that died)

However, this problem lacks information about how many sheep actually died. So it can't accurately give us an answer on how many sheep are left.

But in general if we knew that all but 9 survived then the number of remaining sheep would be 'x+9'.

--- Draft 3 ---
Here's a revised explanation:

### 2.5 Tree‑of‑Thoughts
Tree-of-Thoughts reframes reasoning as a search process rather than a single forward chain.
Instead of producing one linear sequence of thoughts, the model generates multiple candidate thoughts at each step, evaluates their promise, and then expands only the best few. This allows exploration of different reasoning paths before committing to a final answer, similar to how humans brainstorm, prune, and refine ideas.


In this section, you’ll experiment with two simplified versions of ToT:
1. Word Ladder puzzle solver: a small example where each “thought” is a candidate word transition.
2. Generic ToT search (depth 2, width 2): a minimal logic to expand, evaluate, and select reasoning branches

In [6]:
###### Word Ladder Puzzle ##########

def neighbors(word, vocabulary):
    # Generate all valid one-letter mutations of 'word' that exist in 'vocabulary' and return them.
    # Generate all valid one-letter mutations of 'word'
    result = []

    for candidate in vocabulary:
        if len(candidate) != len(word):
            continue

        # Count differing characters
        diff = sum(c1 != c2 for c1, c2 in zip(word, candidate))

        if diff == 1:
            result.append(candidate)

    return result


def tree_of_thought(start, goal, vocab, max_depth=5, beam_width=4):
    # Search over partial thoughts (paths) using a small beam.
    # Step 1: Initialize the frontier with a single path [start]
    # Step 2: For each depth, expand each path by one neighbor from 'neighbors'
    # Step 3: Score paths by edit distance between last word and 'goal' (smaller is better)
    # Step 4: Keep the top 'beam_width' paths and stop early if any reaches 'goal'
    # Step 5: Return the best goal-reaching path or None
    # Step 1: Initialize the frontier
    frontier = [[start]]

    # Step 2: Expand paths
    for _ in range(max_depth):
        new_frontier = []

        for path in frontier:
            last = path[-1]

            # Goal reached
            if last == goal:
                return path

            # Expand neighbors
            for nxt in neighbors(last, vocab):
                if nxt not in path:          # avoid cycles
                    new_frontier.append(path + [nxt])

        if not new_frontier:
            return None

        # Step 3: Score by Hamming (edit) distance to goal
        def score(path):
            last = path[-1]
            return sum(a != b for a, b in zip(last, goal))

        # Step 4: Keep best beam_width paths
        new_frontier.sort(key=score)
        frontier = new_frontier[:beam_width]

    # Step 5: Return goal path if found
    for path in frontier:
        if path[-1] == goal:
            return path

    return None


vocab = {"hit","dot","cog","log","dog","lot","lit","hot"}
print(tree_of_thought("hit", "cog", vocab)) # one candidate solution: ['hit', 'hot', 'dot', 'dog', 'cog']


Task was destroyed but it is pending!
task: <Task pending name='Task-103' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/surajpandey/Documents/Python/web-dev-cohort/genai/venv/lib/python3.12/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-104' coro=<Kernel.shell_main() running at /Users/surajpandey/Documents/Python/web-dev-cohort/genai/venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/surajpandey/Documents/Python/web-dev-cohort/genai/venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py:563]>
/opt/anaconda3/lib/python3.12/collections/__init__.py:447: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  @classmethod
Task was destroyed but it is pending!
task: <Task pending name='Task-104' coro=<Kernel.shell_main() running at /Users/surajpandey/Documents/Python/web-dev-cohort/genai/venv/lib/python3.12/site-packages/ipykern

['hit', 'hot', 'lot', 'log', 'cog']


In [7]:
###### Generic ToT Search ##########

import re

MODEL = "llama3.2:latest"

def propose_thoughts(question, state, k=2):
    # Propose up to k next “thoughts” that extend the current partial solution/state.
    # Steps: build a short prompt with problem + current state; call your client with n=k. Then return a list of stripped strings (≤ k).
    # Propose up to k next thoughts
    prompt = f"""
Problem:
{question}

Current partial solution:
{state if state else 'None'}

Suggest {k} different next reasoning steps or ideas.
Return each idea on a separate line.
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.8,
        n=k
    )

    thoughts = []
    for choice in response.choices:
        thoughts.extend([
            line.strip("-•1234567890. ")
            for line in choice.message.content.splitlines()
            if line.strip()
        ])

    return thoughts[:k]


def score_state(question, state):
    # Score how promising a partial solution is on a 1–10 scale (higher is better).
    # Steps: build a rating prompt; call the model; parse the first integer 1–10;
   # Score a partial solution from 1 to 10
    prompt = f"""
Problem:
{question}

Partial solution:
{state}

Rate this solution from 1 to 10.
Respond with only the number.
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    text = response.choices[0].message.content

    match = re.search(r"\b([1-9]|10)\b", text)
    return int(match.group(1)) if match else 1


def tree_of_thoughts(question, depth=2, width=2):
    # Run a tiny ToT search: expand states with propose_thoughts, score with score_state, keep top-k at each depth.
    # Steps: initialize frontier=[("", 0)]; for each depth, expand each state with k=width thoughts; score each; sort by score desc; keep top 'width'; return best state and score.
    # Initialize frontier
    frontier = [("", 0)]

    for _ in range(depth):
        candidates = []

        for state, _ in frontier:
            thoughts = propose_thoughts(question, state, k=width)

            for thought in thoughts:
                new_state = (state + "\n" + thought).strip()
                score = score_state(question, new_state)
                candidates.append((new_state, score))

        # Keep best states
        candidates.sort(key=lambda x: x[1], reverse=True)
        frontier = candidates[:width]

    return frontier[0]


question = "Design a plan for a weekend science workshop for 12-year-olds."
solution, score = tree_of_thoughts(question)

print(f"Best solution (score {score}):\n{solution}")

Best solution (score 8):
Define the workshop's objectives and goals: Before designing the plan, it's essential to determine what the workshop aims to achieve. What skills or knowledge do you want the 12-year-olds to gain? Will they be learning about science topics such as physics, chemistry, or biology? Are there any specific concepts or themes you want them to explore?
Here are two possible next reasoning steps or ideas for designing the plan:


---  
# 3‑ Training Models for Reasoning

### 3.1: CoT Training
Chain-of-Thought (CoT) training conditions the model on explicit rationales during fine-tuning. Instead of teaching the model to output only the final answer, we train on (question, rationale, answer) so the model learns to internalize multi-step reasoning patterns. A practical recipe is STaR (Self-Taught Reasoner), which uses a stronger teacher model to bootstrap rationales that a smaller student can learn from.

For tasks that require multi-hop reasoning, models fine-tuned on rationales often achieve higher accuracy and are more stable at inference time than models trained on direct answers only. 

Training a full language model is beyond the scope of this notebook, but here is the high-level workflow followed by a short pseudocode:
- Collect questions: Prepare a dataset of questions and correct answers.
- Generate rationales: Use a strong LLM to produce step-by-step reasoning ending with the correct answer.
- Filter and clean: Discard incorrect or low-quality rationales.
- Prepare training data: Format triples (question, rationale, answer) for supervised fine-tuning.
- Fine-tune: Fine-tune the LLM on rationales.
- Iterate: Refine prompts, improve data quality, and retrain for stronger reasoning.

In [ ]:
# Pseudocode (STaR loop)
# for round in 1 ... iters:
    # STEP 1: self-generate reasoning (teacher creates rationale + answer)
    # STEP 2: keep only correct, high-quality traces
    # STEP 3: fine-tune student on (question, rationale, answer) data

### 3.2: ORM vs PRM + RL
Training a Reward Model (RM) allows large language models to be improved through reinforcement learning (RL). Instead of fine-tuning directly on examples, we train a separate model that can score or rank model outputs, and use those scores as feedback signals to refine the policy model.

Two main reward modeling approaches are ORM (predicts a scalar reward for the final answer) and PRM (evaluates the reasoning steps instead of just the outcome)



| Approach | Typical loss | When to use |
|-----------|-------------|-------------|
|*Outcome Reward Model* | Predict scalar reward | Easy to collect training data using verifiers |
|*Process Reward Model* | Predict rewards per step | Difficult to collect training data but more accurate |
| *RLHF* | Use RM as reward in **RL** fine‑tuning | Aligns policy with human signals | Aligns model policy with human or synthetic preferences




In [ ]:
# for round = 1 ... iters:
    # STEP 1:  Generate reasoning
        # sample a minibatch of questions
        # policy roll‑out (actions + log‑probs)
    # STEP 2:  Score the trajectory
        # ORM: scalar reward for the final answer / PRM: scalar reward for the thought process
    # STEP 3:  Reinforce the policy (PPO)

---  
# 4‑ A Deep Research Agent

A deep-research agent pairs a reasoning model (e.g., deepseek-r1) with external tools for web search and retrieval. We will follow the ReAct pattern: the model writes short thoughts, decides when to call tools, reads observations, and continues reasoning until it can answer or reaches a step limit.

We now combine a **search tool** with a reasoning model (e.g., `deepseek-r1`) in a multi-step setup. We follow the *ReAct* pattern (reason → tool → observation):

1. The model reasoins and decides to use tools
2. The agent searches and feed condensed snippets back as context
3. Iterate until the model answers or hits a step limit

We use `AgentType.OPENAI_FUNCTIONS`, which hides the loop inside the LangChain agent.

In [11]:
from ddgs import DDGS
from langchain_core.tools import Tool

def ddg_search(query: str, k: int = 5) -> str:
    # Use DDGS to run a simple web search and return joined snippets.
    with DDGS() as ddgs:
        results = ddgs.text(query, max_results=k)
        snippets = [
            f"{r['title']}: {r['body']}"
            for r in results
        ]
    return "\n".join(snippets)

search_tool = Tool(
    name="DuckDuckGo Search",
    func=ddg_search,
    description="Search the public web. Input: a plain English query. Returns: concatenated snippets."
)


In [13]:
from langchain_classic.agents import initialize_agent, AgentType
from langchain_community.chat_models import ChatOllama

MODEL = "gemma4:e2b"
question = "What are the best resources to learn machine learning in 2025?"

# Step 1: Initialize the reasoning model via ChatOllama
llm = ChatOllama(model=MODEL, temperature=0)

# Step 2: Build the agent with tool access
agent = initialize_agent(
    tools=[search_tool],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Step 3: Ask the question
response = agent.invoke({"input": question})

print("\nFinal Answer:\n")
print(response["output"])



> Entering new AgentExecutor chain...
Thought: I need to find the best, most current resources for learning Machine Learning, keeping in mind that the context is for 2025. This requires searching for current, highly-regarded courses, books, and platforms.

Action:
```json
{
  "action": "DuckDuckGo Search",
  "action_input": "best machine learning resources 2025"
}
```
Observation: Machine Learning for Everybody – Full Course - YouTube: Learn Machine Learning in a way that is accessible to absolute beginners. You will learn the basics of Machine Learning and how to use TensorFlow to implemen...
Best Machine Learning Resources | TikTok: Discover videos related to Best Machine Learning Resources on TikTok. See more videos about Apprendre Lallemand, English Learning, Programmieren Lernen, Quantum Physics for Beginners, Educational Toys, Learning Tips.
Beginner Guide to Machine Learning: Best Books To Learn Machine Learning For Beginners. Learn about Random Forest for Machine Learning | M

# Optional (Multi-agent Deep Research)
Instead of a single multi-step agent, you can design multiple collaborating agents such as a Planner, Searcher, Summarizer, and Verifier that pass information and refine each other’s outputs. This setup improves robustness, diversity of reasoning, and division of labor.

Try building a simple setup with 2–3 agents that share goals and messages, for example Planner → Researcher → Writer.

In [ ]:
def parallel_research(query, n=3):
    # Run n independent research runs in parallel and return their answers.
    # Steps: use ThreadPoolExecutor; submit n calls to your agent/search pipeline; gather results in order.
    """
    YOUR CODE HERE
    """

answers = parallel_research("What are the best resources to learn ML in 2025?")
for i,a in enumerate(answers,1):
    print(f"[Run {i}] {a[:200]}…")

## 🎉 Congratulations!

* Practised various inference‑time reasoning methods
* Gained intuition about training reasoning models
* You have built a **deep-research agent**: reasoning model like deep-seek r1 + ReAct-style agent + tool use (web search)
* Try adding more tools, and extending the deep-research to a multi-agent system: many agents researching web in parallel.


👏 **Great job!** Take a moment to celebrate. The techniques you implemented here power many production agents and chatbots.